# EXP003 — FAISS Index Type vs. Recall

**Question:** Which FAISS index type gives acceptable recall for NetWeave's node embedding similarity search at current and projected node counts?
**Hypothesis:** HNSW gives best recall/speed tradeoff at 5K nodes; flat indices sufficient at current scale.
**Success criterion:** Winning index achieves Recall@10 ≥ 0.90 at 5K nodes with query latency < 10ms.
**Production impact:** `netweave/src/expand.py` → index type selection in v2 build.
**Author:** Chuck Wiley  **Date:** 2026-06-23

In [ ]:
import sys
sys.path.insert(0, ".")

import time
import faiss
import mlflow
import mlflow.tracking
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from lib.niw_graph import load_graph
from lib.niw_mlflow import start_run, log_result, compare_runs

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENT_ID = "EXP003"
EXPERIMENT_NAME = "FAISS Index Type"
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
# Always load from shared/ — never modify source data
G = load_graph(
    nodes_path="shared/nodes.parquet",
    edges_path="shared/edges.parquet"
)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Build embeddings from node text
encoder = SentenceTransformer("all-MiniLM-L6-v2")
node_ids = list(G.nodes())
node_texts = [
    f"{G.nodes[n].get('employer', '')} {G.nodes[n].get('position', '')} "
    f"{' '.join(G.nodes[n].get('domain_tags', []))}"
    for n in node_ids
]
print("Encoding node embeddings...")
embeddings = encoder.encode(node_texts, show_progress_bar=True).astype("float32")
faiss.normalize_L2(embeddings)  # normalize for inner product = cosine
d = embeddings.shape[1]
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
SCALE_SIZES = [500, 1000, 2000, 5000]
N_QUERIES = 20
K = 10

def build_index(index_type: str, vecs: np.ndarray) -> faiss.Index:
    n, d = vecs.shape
    if index_type == "flat_l2":
        idx = faiss.IndexFlatL2(d)
    elif index_type == "flat_ip":
        idx = faiss.IndexFlatIP(d)
    elif index_type == "ivf_flat":
        quantizer = faiss.IndexFlatL2(d)
        idx = faiss.IndexIVFFlat(quantizer, d, min(16, n // 2))
        idx.train(vecs)
    elif index_type == "hnsw":
        idx = faiss.IndexHNSWFlat(d, 32)
    else:
        raise ValueError(f"Unknown index type: {index_type}")
    idx.add(vecs)
    return idx

variants = {
    "flat_l2":  {"index_type": "flat_l2"},
    "flat_ip":  {"index_type": "flat_ip"},
    "ivf_flat": {"index_type": "ivf_flat"},
    "hnsw":     {"index_type": "hnsw"},
}

In [ ]:
# Ground truth: use flat_l2 exact search as oracle
def get_ground_truth(vecs: np.ndarray, query_vecs: np.ndarray, k: int) -> np.ndarray:
    oracle = faiss.IndexFlatL2(vecs.shape[1])
    oracle.add(vecs)
    _, I = oracle.search(query_vecs, k)
    return I

results = []
for scale in SCALE_SIZES:
    # Pad real embeddings with synthetic if needed
    if scale <= len(embeddings):
        vecs = embeddings[:scale]
    else:
        synthetic = np.random.randn(scale - len(embeddings), d).astype("float32")
        faiss.normalize_L2(synthetic)
        vecs = np.vstack([embeddings, synthetic])

    n_q = min(N_QUERIES, len(vecs))
    query_vecs = vecs[:n_q].copy()
    gt = get_ground_truth(vecs, query_vecs, K + 1)[:, 1:]  # exclude self

    for variant_name, params in variants.items():
        with mlflow.start_run(run_name=f"{EXPERIMENT_ID}_{variant_name}_n{scale}"):
            mlflow.log_params({**params, "scale": scale})

            t0 = time.perf_counter()
            idx = build_index(params["index_type"], vecs.copy())
            build_ms = (time.perf_counter() - t0) * 1000

            t0 = time.perf_counter()
            _, I = idx.search(query_vecs, K)
            query_ms = (time.perf_counter() - t0) * 1000 / n_q

            # Recall@K
            recall = np.mean([
                len(set(I[i]) & set(gt[i])) / K
                for i in range(n_q)
            ])

            mlflow.log_metrics({
                f"recall_at_{K}": recall,
                "query_latency_ms": query_ms,
                "build_time_ms": build_ms,
            })
            results.append({
                "variant": variant_name, "scale": scale,
                f"recall@{K}": recall, "latency_ms": query_ms,
            })
            print(f"{variant_name} @{scale}: recall={recall:.4f}, latency={query_ms:.1f}ms")

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for variant, group in results_df.groupby("variant"):
    axes[0].plot(group["scale"], group[f"recall@{K}"], marker="o", label=variant)
    axes[1].plot(group["scale"], group["latency_ms"], marker="o", label=variant)

axes[0].set_xlabel("Node count")
axes[0].set_ylabel(f"Recall@{K}")
axes[0].set_title(f"{EXPERIMENT_ID}: Recall vs. Scale")
axes[0].legend()

axes[1].set_xlabel("Node count")
axes[1].set_ylabel("Query latency (ms)")
axes[1].set_title(f"{EXPERIMENT_ID}: Latency vs. Scale")
axes[1].legend()

plt.tight_layout()
plt.savefig("experiments/EXP003_faiss_index/results.png", dpi=150)
plt.show()

## Result and Decision

**Winner:** [fill in after running]
**Recall@10 at 5K nodes:** [fill in]
**Query latency at 5K nodes:** [fill in ms]
**Decision:** [VALIDATED | INCONCLUSIVE | REJECTED]

**If VALIDATED — production change:**
File: `netweave/src/expand.py`
Function: Index type selection in v2 build.
Change: Use winning index type when building the similarity search index.
Notebook citation: `# Validated in EXP003 — see netweave-lab/experiments/EXP003_faiss_index/`